# 06 · GeoTIFF / Cloud-Optimized GeoTIFF

`da.tp.to_cog(path)` writes a **Cloud-Optimized GeoTIFF** via rioxarray/rasterio (GDAL).
The same dim-detection and 0→360 longitude wrap as `serialize` are applied, and the raster is written
**north-up** (row 0 = north) with `NaN` nodata — so it round-trips through terraplot's `unpackGeoTiff`,
and also opens in QGIS, GDAL, etc.

> Requires `pip install 'pyterraplot[raster]'`. With `path=None` you get the COG **bytes** instead of a file.

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

da = make_field()
da

## Write a COG file

In [ ]:
from pathlib import Path
try:
    import rioxarray  # noqa: F401
    have_raster = True
except ImportError:
    have_raster = False
    print("pyterraplot[raster] not installed — `pip install 'pyterraplot[raster]'` to run this notebook")

if have_raster:
    p = da.tp.to_cog("field.tif")
    print("wrote", p, f"({Path(p).stat().st_size/1024:.0f} kB)")

## Round-trip: read it back

Latitude should come back **descending** (north-up), CRS tagged, NaN nodata preserved.

In [ ]:
if have_raster:
    import rioxarray
    back = rioxarray.open_rasterio("field.tif")  # (band, y, x)
    print("shape :", back.shape)
    print("CRS   :", back.rio.crs)
    print("nodata:", back.rio.nodata)
    print("lat[0], lat[-1]:", float(back.y[0]), float(back.y[-1]), "(descending = north-up)")
    print("nan cells preserved:", int(np.isnan(back.values).sum()))

## In-memory bytes (`path=None`)

Handy for serving a COG straight from a web handler without touching disk.

In [ ]:
if have_raster:
    blob = da.tp.to_cog()  # returns bytes
    print(type(blob), f"{len(blob)/1024:.0f} kB")
    print("TIFF magic:", blob[:4])  # b'II*\x00' (little-endian) or b'MM\x00*'